# Prep ERA5 surface fluxes

In [ ]:
import pathlib
import numpy as np
import xarray as xr
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import datetime
import seaborn as sns
import time
import cmocean
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import scipy.signal
import copy
import tqdm
import os

## (optional) remove gridlines from plots
sns.set(rc={"axes.facecolor": "white", "axes.grid": False})

## specify data path
DATA_FP = pathlib.Path(os.environ["DATA_FP"])

## funcs

In [ ]:
def load_flux_from_fp(fp):
    """load flux data from given filepath"""

    ## get list of varnames
    varname_dict = dict(
        mslhf="Mean surface latent heat flux",
        msshf="Mean surface sensible heat flux",
        msnlwrf="Mean surface net long-wave radiation flux",
        msnlwrfcs="Mean surface net long-wave radiation flux, clear sky",
        msnswrf="Mean surface net short-wave radiation flux",
        msnswrfcs="Mean surface net short-wave radiation flux, clear sky",
    )
    varnames = varname_dict.keys()

    ## empty list to hold data
    data = []

    ## loop through years
    for dir in tqdm.tqdm(sorted(list((fp).glob("*")))):

        ## get list of files to open for given year
        files = []

        ## loop thru variables
        for p in dir.glob("*nc"):

            ## check if file contains given one of given variables
            for v in varnames:
                if v in p.name:
                    files.append(p)

        ## open files which contain variables
        data_ = xr.open_mfdataset(files, combine="nested")

        ## trim it
        data.append(data_.sel(longitude=slice(120, 280), latitude=slice(10, -10)))

    ## concatenate data
    data = xr.concat(data, dim="time")

    return data

## load

### filepaths

In [ ]:
## 1979 onwards
era5_fp0 = pathlib.Path("/gdex/data/d633001/e5.moda.fc.sfc.meanflux")

## pre-1979
era5_fp1 = pathlib.Path("/gdex/data/d633005/e5p.moda.fc.sfc.meanflux")

### do data loading

In [ ]:
## load data
data_post_1978 = load_flux_from_fp(era5_fp0)
data_pre_1979 = load_flux_from_fp(era5_fp1)

## merge
data = xr.concat([data_pre_1979, data_post_1978], dim="time")

## Save to file

In [ ]:
fp = DATA_FP / "era5_nhf_glade.nc"

if fp.is_file():
    pass

else:
    data.to_netcdf(DATA_FP / "era5_nhf_glade.nc")